## 1. Model Evaluation and Explainability.
This notebook evaluates and interprets the performance of the convolutional models (Vitorio Custom CNN, MobileNetV2, EfficientNet-BO, and ResNet-50) trained in `03_model_training.ipynb`.
#### Rationale:
- Quantitatively assess each model's predictive performance using accuracy, precision, recall, F1-score, and AUROC.
- Generate visual explainability results using both **Grad-CAM** (feauture-level attention) and **LIME** (local model-agnostic analysis).
- Compare interpretability methods to undestand each model's decision-making behivour on chest X-rays.
- Ensure consistent evaluation, reproducibility, and visual documentation across models for reporting.

## 2. Environment Setup and Imports.
This section configures the evaluation environment and imports the necessary libraries.
#### Rationale:
- Ensure that the project root directory is visible to Python so that modules under `src/`  
(e.g., `transforms.py`) can be imported reproducibly across notebooks.
- Ensures consistent preprocessing and folder structure inherited from previous notebooks.
- Reuses modular code (`src/`) to prevent redundancy and preserve repoducibility.
- Prepares CUDA availability and output directories for model evaluation and visual explainability.

In [8]:
# Path Setup.
# Extend Python's search path to include the project root, enabling
# reproducible imports from the src/ folder (e.g., src.transforms).
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()     # Step up from notebooks/ to project root.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root added to path: {PROJECT_ROOT}")

# Core scientific visualisation libraries.
import os, json, torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import seaborn as sns

# TorchVision and explainability dependencies.
# Grad-CAM and LIME will be implemented later (Section 5 and 6).
from torchvision import models, transforms
from lime import lime_image # For AI Interpretability's sake.
from PIL import Image

# Reuse project modules from 'scr/' for reproducibility.
from src.transforms import build_shared_transforms # Same transforms used during training.

# Path configuration and runtime device selection.
CHECKPOINTS_DIR = PROJECT_ROOT / "experiments/checkpoints"
RESULTS_DIR     = PROJECT_ROOT / "experiments/results"
POOLS_DIR       = PROJECT_ROOT / "experiments/pools"

DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Evaluation device: {DEVICE.upper()}')

# Ensure required output directories exist.
for folder in [CHECKPOINTS_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
    print(f'Folder ready: {folder}')

# Load preprocessing configuration (reused from previous notebooks).
config_path = POOLS_DIR / "preprocessing_config.json"
if config_path.exists():
    with open(config_path, "r") as f:
        cfg = json.load(f)
    IMG_SIZE = int(cfg["img_size"])
    IMAGENET_MEAN = cfg["imagenet_mean"]
    IMAGENET_STD = cfg["imagenet_std"]
    print(f'Loaded preprocessing parameters from: {config_path.relative_to(PROJECT_ROOT)}')
else:
    IMG_SIZE = 256
    IMAGENET_MEAN = [0.486, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.255]
    print("No preprocessing_config.json found. Using default ImageNet statistics.")

# Build evaluation transforms (same as in training phase).
_, eval_transforms = build_shared_transforms(
    img_size=IMG_SIZE,
    imagenet_mean=IMAGENET_MEAN,
    imagenet_std=IMAGENET_STD,
)
print(f'Transforms ready ({IMG_SIZE}x{IMG_SIZE}, ImageNet-normalised).')

Project root added to path: /lab/px/msc-sept-24-ft-cohort-capstone-chrisanich
Evaluation device: CUDA
Folder ready: /lab/px/msc-sept-24-ft-cohort-capstone-chrisanich/experiments/checkpoints
Folder ready: /lab/px/msc-sept-24-ft-cohort-capstone-chrisanich/experiments/results
Loaded preprocessing parameters from: experiments/pools/preprocessing_config.json
Transforms ready (256x256, ImageNet-normalised).
